In [ ]:
from google.colab import drive
import zipfile
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define paths - UPDATE 'intel_image_dataset.zip' to your exact Drive file name
zip_path = '/content/drive/MyDrive/intel_image_dataset.zip'
extract_dir = '/content/dataset'

# Extract the dataset locally to Colab
if not os.path.exists(extract_dir):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print("Dataset successfully extracted to Colab!")
else:
    print("Dataset already extracted.")

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Point this to the folder containing the 6 class subdirectories (buildings, forest, etc.)
# If your zip extracts a parent folder first (e.g., /content/dataset/dataset/), update this path.
data_dir = '/content/dataset' 

# Standardize image dimensions for VGG16
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32

# 1. Training Generator WITH Data Augmentation and Normalization
train_datagen = ImageDataGenerator(
    rescale=1./255,             # Normalize pixels to 0-1
    rotation_range=20,          # Augmentation: Rotation
    width_shift_range=0.2,      # Augmentation: Translation
    height_shift_range=0.2,     # Augmentation: Translation
    zoom_range=0.2,             # Augmentation: Zoom
    horizontal_flip=True,       # Augmentation: Horizontal Flip
    validation_split=0.30       # Reserve 30% for Validation/Test combined
)

# 2. Validation/Test Generator with ONLY Normalization (No augmentation)
test_val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.30
)

# Create Training Dataset (70% of data)
train_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

# Create Validation Dataset (15% of data)
# Note: We split the remaining 30% into two halves by controlling the flow in later steps, 
# or you can use this generator directly for validation.
validation_generator = test_val_datagen.flow_from_directory(
    data_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False # Keep false for stable evaluation
)

print("\nClass mapping:", train_generator.class_indices)